# Access Requests with the ML App Python client

A focused, rerunnable walkthrough of Business Case discovery, auditable access requests and bounded request queues. It does not expose inaccessible Business Case details.


In [1]:
from pathlib import Path
import sys

repository_root = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'ml_app_client').is_dir()), None)
if repository_root is None:
    raise RuntimeError('Start Jupyter inside the ml-app repository')
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from ml_app_client import MLAppClient

client = MLAppClient.connect()
print('Connected as', client.me().get('login_name'))


Connected as maksiu2@wp.pl


## Find a Business Case without reading its details

The organization-wide directory is a bounded, minimal projection. Select an entry before requesting access; do not infer case details from the directory.


In [7]:
SEARCH = 'churn'
directory = client.page_business_case_catalog(search=SEARCH, limit=30)
for bc in directory.items:
    client.display_object(bc)

Id,413a4d16-235f-48d7-957b-cc02592e7ef0
Name,[MLAPP client module] Customer churn demo
Status,active
Access Role,—
Request Status,—


Id,1960dd86-569a-4f63-bb8e-949a6b547e31
Name,Storage Subscription Churn
Status,draft
Access Role,—
Request Status,—


## Submit a request deliberately

Uncomment this only after choosing an entry you do not already have access to. Explain the work that needs the requested role. `owner` is never requestable; ownership transfers use the separate audited Business Case workflow.


In [ ]:
entry = directory.items[0]
request = client.request_business_case_access(
    entry.id,
    requested_role='manager',
    justification='Just give me',
)

ConflictError: POST /sharing/business-cases/413a4d16-235f-48d7-957b-cc02592e7ef0/access-requests returned HTTP 409: A pending access request already exists for this Business Case

In [10]:
sink = client.display_object(request)

Id,beb5f2d9-1976-4824-8408-5770efc27125
Business Case Id,413a4d16-235f-48d7-957b-cc02592e7ef0
Status,pending
Requested Role,manager


## Inspect only the queue you need

`incoming` is for cases you can manage; `mine` is your submitted queue. Both are bounded pages. Use `submitted_history` or `handled` when reviewing completed activity.


In [ ]:
submitted = client.page_business_case_access_requests(box='mine', limit=30)
sink = client.display_object(submitted.items[0])

Id,beb5f2d9-1976-4824-8408-5770efc27125
Business Case Id,413a4d16-235f-48d7-957b-cc02592e7ef0
Status,pending
Requested Role,manager


## Decide an incoming request

A manager can approve a pending request and grant an allowed role atomically, or reject it with a recorded note. The selected role cannot exceed the manager's effective role.


In [15]:
client = MLAppClient.connect()
print('Connected as', client.me().get('login_name'))

Connected as test3@example.pl


In [17]:
incoming = client.page_business_case_access_requests(box='incoming', limit=30)
sink = client.display_object(incoming.items[0])

Id,beb5f2d9-1976-4824-8408-5770efc27125
Business Case Id,413a4d16-235f-48d7-957b-cc02592e7ef0
Status,pending
Requested Role,manager


In [ ]:
request = incoming.items[0]

rejected = client.reject_business_case_access_request(
    request.id, decision_note='Access is not required for this work.',
)
sink = client.display_object(rejected)


Id,beb5f2d9-1976-4824-8408-5770efc27125
Business Case Id,413a4d16-235f-48d7-957b-cc02592e7ef0
Status,rejected
Requested Role,manager


## Example of another, accepted request

In [19]:
client = MLAppClient.connect()
print('Connected as', client.me().get('login_name'))

Connected as maksiu2@wp.pl


In [20]:
SEARCH = 'churn'
directory = client.page_business_case_catalog(search=SEARCH, limit=30)
for bc in directory.items:
    client.display_object(bc)

Id,413a4d16-235f-48d7-957b-cc02592e7ef0
Name,[MLAPP client module] Customer churn demo
Status,active
Access Role,—
Request Status,—


Id,1960dd86-569a-4f63-bb8e-949a6b547e31
Name,Storage Subscription Churn
Status,draft
Access Role,—
Request Status,—


In [21]:
entry = directory.items[0]
request = client.request_business_case_access(
    entry.id,
    requested_role='reader',
    justification='Let me read I will not change anything',
)
sink = client.display_object(request)

Id,f8719053-9393-4e09-a691-08d9c0fee8f6
Business Case Id,413a4d16-235f-48d7-957b-cc02592e7ef0
Status,pending
Requested Role,reader


## Relog to the BC's owner

In [23]:
client = MLAppClient.connect()
print('Connected as', client.me().get('login_name'))

Connected as test3@example.pl


In [26]:
incoming = client.page_business_case_access_requests(box='incoming', limit=30)
sink = client.display_object(incoming.items[0])

Id,f8719053-9393-4e09-a691-08d9c0fee8f6
Business Case Id,413a4d16-235f-48d7-957b-cc02592e7ef0
Status,pending
Requested Role,reader


In [28]:
request = incoming.items[0]

approved = client.approve_business_case_access_request(
    request.id, access_role='reader', decision_note='Approved for reporting.',
)
sink = client.display_object(approved)

Id,f8719053-9393-4e09-a691-08d9c0fee8f6
Business Case Id,413a4d16-235f-48d7-957b-cc02592e7ef0
Status,approved
Requested Role,reader
